# Bulk RNA-seq WGCNA Template

用于 normalized expression 矩阵的共表达模块构建、模块-性状关联、hub gene 导出。建议输入 VST/rlog/log2(TPM+1) 后的表达矩阵，不要直接使用 raw counts。

## 1. Parameter Configuration

In [ ]:
# ===================== Parameter Configuration =====================
EXPR_FILE <- "./1-DEG/vsd_matrix.csv"       # genes x samples
TRAIT_FILE <- "./1-DEG/colData.csv"           # exported by RNAseq_General; rows are samples or contains SAMPLE_COLUMN
GENE_COLUMN <- NULL
SAMPLE_COLUMN <- "sample"
GROUP_COLUMN <- "condition"

MIN_MAD_QUANTILE <- 0.5                       # keep top variable genes by MAD
NETWORK_TYPE <- "signed"                      # "signed" recommended for biology
POWER_VECTOR <- c(1:10, seq(12, 30, 2))
MIN_MODULE_SIZE <- 30
MERGE_CUT_HEIGHT <- 0.25
TARGET_MODULES <- NULL                         # e.g. c("blue", "turquoise"); NULL = all significant modules
OUTDIR <- "RNAseq_WGCNA_Output"
dir.create(OUTDIR, showWarnings = FALSE, recursive = TRUE)


## 2. Environment

In [ ]:
options(stringsAsFactors = FALSE)
# First run if needed:
# install.packages(c("WGCNA", "tidyverse", "pheatmap"))

suppressPackageStartupMessages({
  library(WGCNA)
  library(tidyverse)
  library(pheatmap)
})
allowWGCNAThreads()

LIB_DIR <- if (dir.exists("RNAseq_lib")) "RNAseq_lib" else "../RNAseq_lib"
source(file.path(LIB_DIR, "plot_utils.R"))
source(file.path(LIB_DIR, "data_utils.R"))
theme_set(theme_publication())
cat("RNAseq_lib:", LIB_DIR, "\n")

## 3. Load Expression and Traits

In [ ]:
# Load expression matrix and traits with validation
expr <- read_expression_matrix(EXPR_FILE, gene_column = GENE_COLUMN)

traits <- read_metadata(
  TRAIT_FILE,
  sample_column = SAMPLE_COLUMN,
  required_columns = NULL,
  group_column = GROUP_COLUMN
)

validate_samples_match(colnames(expr), traits[[SAMPLE_COLUMN]], strict_order = TRUE)
expr <- expr[, traits[[SAMPLE_COLUMN]], drop = FALSE]
rownames(traits) <- traits[[SAMPLE_COLUMN]]

# WGCNA requires variance-stabilized/normalized expression derived from raw counts.
scale_info <- validate_expression_contract(expr, expected = "vst")

cat("Expression:", nrow(expr), "genes x", ncol(expr), "samples\n")

## 4. Gene Filtering and Sample QC

In [ ]:
gene_mad <- apply(expr, 1, mad, na.rm = TRUE)
expr <- expr[gene_mad >= quantile(gene_mad, MIN_MAD_QUANTILE, na.rm = TRUE), , drop = FALSE]
datExpr <- t(as.matrix(expr))

gsg <- goodSamplesGenes(datExpr, verbose = 3)
if (!gsg$allOK) {
  datExpr <- datExpr[gsg$goodSamples, gsg$goodGenes]
  traits <- traits[rownames(datExpr), , drop = FALSE]
}

sampleTree <- hclust(dist(datExpr), method = "average")
plot_wgcna_sample_tree_pdf(sampleTree, file.path(OUTDIR, "Sample_clustering.pdf"))
cat("WGCNA input:", nrow(datExpr), "samples x", ncol(datExpr), "genes\n")

## 5. Soft Threshold Selection

In [ ]:
sft <- pickSoftThreshold(datExpr, powerVector = POWER_VECTOR, networkType = NETWORK_TYPE, verbose = 5)
soft_power <- sft$powerEstimate
if (is.na(soft_power)) {
  fit_df <- sft$fitIndices
  soft_power <- fit_df$Power[which.max(fit_df$SFT.R.sq)]
  message("No automatic power estimate; using max scale-free fit power: ", soft_power)
}

plot_wgcna_soft_threshold_pdf(sft, selected_power = soft_power,
                                  filename = file.path(OUTDIR, "Soft_threshold_selection.pdf"))
cat("Selected soft power:", soft_power, "\n")


## 6. Network Construction and Module Detection

In [ ]:
net <- blockwiseModules(
  datExpr,
  power = soft_power,
  networkType = NETWORK_TYPE,
  TOMType = NETWORK_TYPE,
  minModuleSize = MIN_MODULE_SIZE,
  reassignThreshold = 0,
  mergeCutHeight = MERGE_CUT_HEIGHT,
  numericLabels = TRUE,
  pamRespectsDendro = FALSE,
  saveTOMs = FALSE,
  verbose = 3
)
moduleColors <- labels2colors(net$colors)
MEs <- orderMEs(net$MEs)

write.csv(data.frame(gene = colnames(datExpr), module = moduleColors), file.path(OUTDIR, "WGCNA_gene_modules.csv"), row.names = FALSE)
saveRDS(list(net = net, moduleColors = moduleColors, MEs = MEs, datExpr = datExpr, traits = traits), file.path(OUTDIR, "WGCNA_network.rds"))

plot_wgcna_module_dendrogram_pdf(net, moduleColors, file.path(OUTDIR, "Module_dendrogram.pdf"))
print(table(moduleColors))


## 7. Module-Trait Correlation

In [ ]:
trait_numeric <- encode_wgcna_traits(traits, sample_column = SAMPLE_COLUMN)
if (ncol(trait_numeric) == 0) stop("No varying non-identifier traits available for WGCNA correlation.")
moduleTraitCor <- cor(MEs, trait_numeric, use = "p")
moduleTraitP <- corPvalueStudent(moduleTraitCor, nrow(datExpr))

write.csv(moduleTraitCor, file.path(OUTDIR, "Module_trait_correlation.csv"))
write.csv(moduleTraitP, file.path(OUTDIR, "Module_trait_pvalue.csv"))

plot_wgcna_module_trait_heatmap_pdf(moduleTraitCor, moduleTraitP,
                                      file.path(OUTDIR, "Module_trait_heatmap.pdf"))


## 8. Hub Gene Export

In [ ]:
gene_module <- data.frame(gene = colnames(datExpr), module = moduleColors)
all_modules <- unique(moduleColors[moduleColors != "grey"])
if (!is.null(TARGET_MODULES)) all_modules <- intersect(all_modules, TARGET_MODULES)

hub_list <- list()
for (mod in all_modules) {
  mod_genes <- gene_module$gene[gene_module$module == mod]
  ME <- MEs[[paste0("ME", mod)]]
  kME <- cor(datExpr[, mod_genes, drop = FALSE], ME, use = "p")
  hub <- data.frame(gene = mod_genes, module = mod, kME = as.numeric(kME)) %>% arrange(desc(abs(kME)))
  hub_list[[mod]] <- hub
  write.csv(hub, file.path(OUTDIR, paste0("Hub_genes_", mod, ".csv")), row.names = FALSE)
}
write.csv(bind_rows(hub_list), file.path(OUTDIR, "Hub_genes_all_modules.csv"), row.names = FALSE)
writeLines(capture.output(sessionInfo()), file.path(OUTDIR, "sessionInfo.txt"))
